# Statistical analysis: are the reliability differences real?

This notebook tests the patterns the dashboards show, using the exported marts and a random
flight-level sample (`data/analysis/flights_sample.parquet`, created by `python -m pipeline.export`).

**Read before interpreting.** With hundreds of thousands of flights, almost any difference is
"statistically significant". So every test below also reports an **effect size** (how big the
difference is) and a confidence interval. Judge findings by effect size and practical impact,
not by p-values alone. Everything here is observational: these tests show association, not cause.

Run top to bottom: *Kernel -> Restart & Run All*.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
MARTS, ANALYSIS = ROOT / "data" / "marts", ROOT / "data" / "analysis"

def load(folder, name):
    parquet = folder / f"{name}.parquet"
    frame = pd.read_parquet(parquet) if parquet.exists() else pd.read_csv(folder / f"{name}.csv")
    for col in frame.columns:  # CSV exports from psql write booleans as t/f
        if pd.api.types.is_string_dtype(frame[col]) and set(frame[col].dropna().unique()) <= {"t", "f"}:
            frame[col] = frame[col].map({"t": True, "f": False})
    return frame

manifest = pd.read_json(MARTS / "manifest.json", typ="series")
if manifest.get("synthetic_data"):
    print("WARNING: SYNTHETIC TEST DATA - results below are meaningless. Load real BTS data first.")

carrier_monthly = load(MARTS, "mart_carrier_monthly")
sample = load(ANALYSIS, "flights_sample")
completed = sample[sample["is_completed"].astype(bool)].copy()
print(f"sample: {len(sample):,} flights ({len(completed):,} completed), "
      f"months {manifest.get('first_month')}-{manifest.get('latest_month')}")

sample: 500,269 flights (489,254 completed), months 202509-202608


## 1. Is on-time performance independent of the carrier? (chi-square test)

Uses the **full** data (all flights, from `mart_carrier_monthly`), not the sample.
Effect size: **Cramer's V** (0 = no association, ~0.1 small, ~0.3 medium, ~0.5 large).

In [2]:
by_carrier = carrier_monthly.groupby("carrier_code")[["on_time_flights", "scheduled_flights"]].sum()
table = np.column_stack([by_carrier["on_time_flights"],
                         by_carrier["scheduled_flights"] - by_carrier["on_time_flights"]])
chi2, p_value, dof, _ = stats.chi2_contingency(table)
n = table.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

summary = (by_carrier.assign(on_time_rate=lambda d: d.on_time_flights / d.scheduled_flights)
           .sort_values("on_time_rate"))
print(f"chi-square = {chi2:,.1f}, dof = {dof}, p = {p_value:.2e}, Cramer's V = {cramers_v:.3f}")
summary.style.format({"on_time_rate": "{:.1%}", "on_time_flights": "{:,}", "scheduled_flights": "{:,}"})

chi-square = 36,723.4, dof = 13, p = 0.00e+00, Cramer's V = 0.072


,on_time_flights,scheduled_flights,on_time_rate
carrier_code,,,
B6,"164,785","240,615",68.5%
NK,"68,574","99,981",68.6%
OH,"169,447","242,979",69.7%
F9,"151,405","213,022",71.1%
AA,"720,560","988,892",72.9%
G4,"92,137","126,005",73.1%
YX,"270,862","364,452",74.3%
WN,"1,041,149","1,391,794",74.8%
OO,"658,960","855,406",77.0%


## 2. Do evening departures arrive later than morning departures? (Welch's t-test)

Welch's version does not assume equal variances (evening delays vary much more).
Effect size: **Cohen's d** (~0.2 small, ~0.5 medium, ~0.8 large). Delay minutes are heavily
right-skewed, so a Mann-Whitney test is shown as a robustness check.

In [3]:
morning = completed.loc[completed.time_of_day == "morning", "arr_delay_min"].dropna()
evening = completed.loc[completed.time_of_day == "evening", "arr_delay_min"].dropna()

t_stat, p_t = stats.ttest_ind(evening, morning, equal_var=False)
pooled_sd = np.sqrt((morning.var(ddof=1) + evening.var(ddof=1)) / 2)
cohens_d = (evening.mean() - morning.mean()) / pooled_sd
diff = evening.mean() - morning.mean()
se = np.sqrt(morning.var(ddof=1) / len(morning) + evening.var(ddof=1) / len(evening))
u_stat, p_u = stats.mannwhitneyu(evening, morning, alternative="two-sided")

print(f"morning mean {morning.mean():.1f} min (n={len(morning):,}) | evening mean {evening.mean():.1f} min (n={len(evening):,})")
print(f"difference {diff:.1f} min, 95% CI [{diff - 1.96 * se:.1f}, {diff + 1.96 * se:.1f}]")
print(f"Welch t = {t_stat:.1f}, p = {p_t:.2e}, Cohen's d = {cohens_d:.3f}")
print(f"Mann-Whitney U p = {p_u:.2e} (median {morning.median():.0f} vs {evening.median():.0f} min)")

morning mean 0.9 min (n=206,512) | evening mean 17.0 min (n=123,448)
difference 16.1 min, 95% CI [15.6, 16.6]
Welch t = 69.8, p = 0.00e+00, Cohen's d = 0.259
Mann-Whitney U p = 0.00e+00 (median -9 vs -1 min)


## 3. Does a late inbound aircraft predict a late flight? (cascade / propagation)

Compares the share of flights arriving 15+ minutes late when the **same aircraft's previous
leg** arrived 15+ late versus on time. Effect size: **risk ratio** with a 95% CI.

In [4]:
legs = completed.dropna(subset=["prev_leg_arr_delay_min"]).copy()
legs["prev_late"] = legs["prev_leg_arr_delay_min"] >= 15
legs["late"] = legs["is_arr_delayed_15"].astype(bool)

a = legs.loc[legs.prev_late, "late"]; b = legs.loc[~legs.prev_late, "late"]
risk_late, risk_ok = a.mean(), b.mean()
rr = risk_late / risk_ok
se_log_rr = np.sqrt((1 - risk_late) / a.sum() + (1 - risk_ok) / b.sum())
ci = np.exp(np.log(rr) + np.array([-1.96, 1.96]) * se_log_rr)
print(f"P(late | previous leg late) = {risk_late:.1%} (n={len(a):,})")
print(f"P(late | previous leg on time) = {risk_ok:.1%} (n={len(b):,})")
print(f"risk ratio = {rr:.2f}, 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]")
legs.groupby(pd.cut(legs.prev_leg_arr_delay_min, [-np.inf, 0, 15, 30, 60, 120, np.inf]),
             observed=True)["late"].agg(["mean", "size"]).rename(columns={"mean": "share_late"})

P(late | previous leg late) = 65.9% (n=73,226)
P(late | previous leg on time) = 15.2% (n=292,139)
risk ratio = 4.33, 95% CI [4.29, 4.37]


,share_late,size
prev_leg_arr_delay_min,,
"(-inf, 0.0]",0.133899,237507
"(0.0, 15.0]",0.236386,57055
"(15.0, 30.0]",0.455055,25175
"(30.0, 60.0]",0.733309,21658
"(60.0, 120.0]",0.867130,14518
"(120.0, inf]",0.799937,9452


## 4. What predicts a 15+ minute arrival delay? (logistic regression)

One model with several factors at once, so each odds ratio is "holding the others fixed".
Odds ratio > 1 = higher odds of delay than the reference group. Reference groups: the first
category alphabetically (shown in the variable name).

In [5]:
import statsmodels.formula.api as smf

model_data = completed.dropna(subset=["is_arr_delayed_15"]).copy()
model_data["delayed"] = model_data["is_arr_delayed_15"].astype(int)
model_data["hub"] = model_data["origin_is_hub"].astype(int)
model_data["month"] = model_data["month"].astype(str)

model = smf.logit(
    "delayed ~ C(time_of_day, Treatment('morning')) + C(carrier_code) + C(month)"
    " + C(distance_band) + hub",
    data=model_data,
).fit(disp=False)

odds = pd.DataFrame({
    "odds_ratio": np.exp(model.params),
    "ci_low": np.exp(model.conf_int()[0]),
    "ci_high": np.exp(model.conf_int()[1]),
    "p_value": model.pvalues,
}).drop(index="Intercept")
print(f"n = {int(model.nobs):,}, pseudo R-squared = {model.prsquared:.3f}")
odds.sort_values("odds_ratio", ascending=False).round(3)

n = 489,254, pseudo R-squared = 0.041


,odds_ratio,ci_low,ci_high,p_value
"C(time_of_day, Treatment('morning'))[T.evening]",2.704,2.658,2.751,0.000
"C(time_of_day, Treatment('morning'))[T.afternoon]",2.093,2.058,2.129,0.000
"C(time_of_day, Treatment('morning'))[T.night]",1.906,1.835,1.980,0.000
C(month)[T.7],1.566,1.515,1.620,0.000
C(month)[T.8],1.455,1.407,1.506,0.000
C(month)[T.12],1.403,1.356,1.452,0.000
C(month)[T.6],1.341,1.296,1.388,0.000
C(carrier_code)[T.NK],1.250,1.181,1.324,0.000
C(month)[T.3],1.229,1.188,1.272,0.000
hub,1.222,1.203,1.241,0.000


## 5. Findings (Sep 2025 – Aug 2026 data; numbers from the outputs above)

1. **A late inbound aircraft is the strongest delay signal.** When the same aircraft's previous leg arrived 15+ minutes late, 65.9% of next flights arrived late, against 15.2% after an on-time previous leg. That is a risk ratio of **4.33** (95% CI 4.29–4.37). The effect grows with the size of the inbound delay: 45.5% late after a 15–30 minute inbound delay, 86.7% after 60–120 minutes.
   *Limits:* association only. Days with bad weather or congestion make both legs late.

2. **Evening departures arrive much later than morning ones.** Mean arrival delay was 17.0 min for evening flights against 0.9 min for morning flights, a difference of **16.1 minutes** (95% CI 15.6–16.6). Cohen's d is **0.26**, a small effect on individual flights because delays are very skewed (medians: −9 min morning, −1 min evening). Holding carrier, month, distance and hub fixed, evening flights have **2.70×** the odds of a 15+ minute delay compared with morning flights (95% CI 2.66–2.75).
   *Limits:* time of day is tied to an aircraft's position in its daily rotation, so part of the evening effect is the cascade from finding 1.

3. **Airlines differ, but less than timing does.** The chi-square test of carrier vs on-time (χ² = 36,723, 13 df) is significant, as anything is at 7 million flights, but **Cramér's V = 0.072**, a small association. In the regression, carrier odds ratios range from 0.69 (Delta) to 1.25 (Spirit) relative to American, all well below the time-of-day effect.

4. **Summer and holiday months carry the most delay risk.** Compared with January, July has 1.57× the odds of a 15+ minute delay (95% CI 1.52–1.62), August 1.46× and December 1.40×. September is the safest month (0.74×). Flights from the 30 hub airports have 1.22× the odds of non-hub flights.

The model's pseudo R² is only 0.041. These factors shift the odds of delay but explain little of which individual flight ends up late, which is expected for operational data like this.
